# Stage 6 — A/B/C Variant Comparison (Graphic Design / Service Resume)

**Represents 5 of the 20 required test cases** (this resume × 5 jobs). Third of four resume-based batches (Trixie Mok, Alex Chen, John Doe, +1 more), 5 test cases each, totalling 20.

Resume tested: `John Doe.pdf` — BS in Graphic Design (2011), but every job since has been retail/service (Sales Associate, Spa Consultant, Fashion Representative). No professional design or technical experience at all. This is the hardest test set: zero real skill overlap with any of the 5 jobs tested.

- **A — Minimal LLM**: bare prompt, "Rewrite this resume for this job."
- **B — Simplified system**: "Extract relevant skills and tailor this resume to the job" — no schema, no anti-fabrication instructions.
- **C — Full system**: the actual deployed app logic (`JobPortalService._generate_tailored_resume`), structured JSON schema, evidence required per claim, explicit instruction never to invent credentials/employers/dates/metrics/skills.

Same 5 jobs and 4 criteria as the Trixie and fresh-grad tests, for direct comparability.

**This is a retest.** The results below are current — regenerated after the team shipped a deterministic anti-fabrication check to System C (`_enforce_fidelity`), the exact fix this test's original run had recommended. The original pre-fix results are preserved as `*_PREFIX.json` for direct before/after comparison (see the Retest section near the end).

In [1]:
import json
from pathlib import Path

HERE = Path(".")
naive = json.loads((HERE / "variant_comparison_results.json").read_text(encoding="utf-8"))
grounded = json.loads((HERE / "variant_comparison_results_grounded_judge.json").read_text(encoding="utf-8"))

CRITERIA = ["grounding", "personalization", "correctness", "clarity"]
print(f"Loaded {len(naive)} test cases (naive judge) and {len(grounded)} (grounding-gated judge)")

Loaded 5 test cases (naive judge) and 5 (grounding-gated judge)


## Round 1: Naive LLM-judge scoring

In [2]:
def print_table(results, score_key):
    header = f"{'Job':<22}{'Variant':<8}" + "".join(f"{c[:10]:<12}" for c in CRITERIA) + "avg"
    print(header)
    print("-" * len(header))
    totals = {v: {c: [] for c in CRITERIA} for v in "ABC"}
    for entry in results:
        scores_block = entry[score_key]
        for v in "ABC":
            row = scores_block[v]
            vals = [row[c]["score"] for c in CRITERIA]
            avg = sum(vals) / len(vals)
            for c, val in zip(CRITERIA, vals):
                totals[v][c].append(val)
            print(f"{entry['job_title'][:21]:<22}{v:<8}" + "".join(f"{val:<12}" for val in vals) + f"{avg:.2f}")
    print()
    print("OVERALL AVERAGES")
    for v in "ABC":
        per_c = {c: sum(totals[v][c]) / len(totals[v][c]) for c in CRITERIA}
        overall = sum(per_c.values()) / len(per_c)
        print(f"  {v}: " + ", ".join(f"{c}={val:.2f}" for c, val in per_c.items()) + f"  -> overall={overall:.2f}")

print_table(naive, "scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       1           2           1           2           1.50
Software Engineer     B       3           4           3           4           3.50
Software Engineer     C       5           3           5           5           4.50
Machine Learning      A       1           2           1           3           1.75
Machine Learning      B       2           3           2           3           2.50
Machine Learning      C       4           1           4           4           3.25
DevOps Engineer       A       1           2           3           4           2.50
DevOps Engineer       B       1           2           3           3           2.25
DevOps Engineer       C       1           1           4           2           2.00
Database Administrato A       2           4           3           4           3.25
Databa

Unlike the pre-fix run (where B ranked highest and C lowest), this naive-judge pass now ranks **C highest even without the grounding-gate rule** — a side effect of A/B's raw generations this run being more heavily fabricated than last time. This run-to-run swing is itself evidence that a naive judge alone isn't reliable: A/B's rankings move with how much the unguarded LLM happens to invent on a given call, not with any real quality difference.

## Round 2: Grounding-gated judge (the fix)

Same hard rule as the other two tests: any fabricated skill/employer/metric caps that variant's scores, regardless of fluency.

In [3]:
print_table(grounded, "grounded_judge_scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       1           2           2           2           1.75
Software Engineer     B       1           2           2           2           1.75
Software Engineer     C       5           3           5           4           4.25
Machine Learning      A       1           2           2           2           1.75
Machine Learning      B       1           2           2           2           1.75
Machine Learning      C       1           2           2           2           1.75
DevOps Engineer       A       5           4           5           5           4.75
DevOps Engineer       B       5           3           5           4           4.25
DevOps Engineer       C       5           2           5           5           4.25
Database Administrato A       5           4           5           5           4.75
Databa

In [4]:
# Fabrications the grounding-gated judge actually caught, per variant
for entry in grounded:
    fab = entry["grounded_judge_scores"].get("fabrications", {})
    if any(fab.get(v) for v in "ABC"):
        print(entry["job_title"], "@", entry["company_name"])
        for v in "ABC":
            items = [i for i in (fab.get(v) or []) if i]
            if items:
                print(f"  {v}: {items}")
        print()

Software Engineer @ WestGate Networks
  A: ["Bachelor's Degree in Computer Science or related field", 'Software Engineer Intern', 'Freelance Developer', 'Certified Scrum Master (CSM)', 'Certified Java Developer (OCPJP)']
  B: ['Programming Principles', 'Data Structures', 'Human-Computer Interaction']

Machine Learning @ LumaCore Data
  A: ['Data Visualization', 'Statistical Analysis', 'Computer Programming (Python, R, etc.)', 'Data analysis and visualization', 'Statistical modeling and machine learning (basic knowledge)', 'Programming languages (Python, R, etc.)']
  B: ['Proficient in [insert any relevant software or tools]']
  C: ['Merchandising']

DevOps Engineer @ CloudHarbor Labs

Database Administrator @ Solstice Digital

Full Stack Developer @ InnoWave Networks
  A: ['Programming languages: JavaScript, Node.js, Java', 'Front-end frameworks: AngularJS', 'Back-end frameworks: Express', 'Databases: MongoDB, Postgres, Redis', 'Messaging queues: RabbitMQ', 'Search engines: Elasticsear

## Retest: Before vs. After the Anti-Fabrication Fix

The original run found C added 3 stray unverified words ("linux," "python," "testing") on the DevOps Engineer case. The team's fix (`_enforce_fidelity`) targeted exactly this. Retesting the identical case below confirms that specific fabrication is gone — but also surfaces a new, more interesting problem on a different job.

In [5]:
prefix_grounded = json.loads((HERE / "variant_comparison_results_grounded_judge_PREFIX.json").read_text(encoding="utf-8"))

def overall_by_variant(results, score_key):
    out = {}
    for v in "ABC":
        vals = []
        for entry in results:
            row = entry[score_key][v]
            vals.append(sum(row[c]["score"] for c in CRITERIA) / len(CRITERIA))
        out[v] = sum(vals) / len(vals)
    return out

before = overall_by_variant(prefix_grounded, "grounded_judge_scores")
after = overall_by_variant(grounded, "grounded_judge_scores")

print(f"{'Variant':<20}{'Before fix':<14}{'After fix':<14}")
for v, label in zip("ABC", ["A - Minimal", "B - Simplified", "C - Full system"]):
    print(f"{label:<20}{before[v]:<14.2f}{after[v]:<14.2f}")

print()
print("Per-job C grounding-gated overall, before vs. after:")
before_by_job = {e["job_title"]: sum(e["grounded_judge_scores"]["C"][c]["score"] for c in CRITERIA) / 4 for e in prefix_grounded}
after_by_job = {e["job_title"]: sum(e["grounded_judge_scores"]["C"][c]["score"] for c in CRITERIA) / 4 for e in grounded}
for job in before_by_job:
    print(f"  {job:<24} before={before_by_job[job]:<6.2f} after={after_by_job.get(job, float('nan')):<6.2f}")

Variant             Before fix    After fix     
A - Minimal         2.80          2.95          
B - Simplified      2.80          3.35          
C - Full system     3.55          3.55          

Per-job C grounding-gated overall, before vs. after:
  Software Engineer        before=3.50   after=4.25  
  Machine Learning         before=4.00   after=1.75  
  DevOps Engineer          before=1.75   after=4.25  
  Database Administrator   before=4.00   after=3.50  
  Full Stack Developer     before=4.50   after=4.00  


In [6]:
# Investigate the new flag on C for Machine Learning: is "Merchandising" actually in the delivered resume?
ml_entry = [e for e in naive if e["job_title"] == "Machine Learning"][0]
c_out = ml_entry["outputs"]["C"]

print("Grounding reason (grounded judge):")
ml_grounded = [e for e in grounded if e["job_title"] == "Machine Learning"][0]
print(" ", ml_grounded["grounded_judge_scores"]["C"]["grounding"])
print()
print("Is 'Merchandising' in the delivered resume text?", "Merchandising" in c_out["rewritten_text"])
print("Is 'Merchandising' in the delivered skills list?", "Merchandising" in c_out["resume"].get("skills", []))
print()
print("System C's own validation_results (the audit trail the judge was also shown):")
for v in c_out["validation_results"]:
    print(" ", v)

Grounding reason (grounded judge):
  {'score': 1, 'reason': 'Fabricated skill'}

Is 'Merchandising' in the delivered resume text? False
Is 'Merchandising' in the delivered skills list? False

System C's own validation_results (the audit trail the judge was also shown):
  {'status': 'fail', 'rule': 'relevant experience', 'claim': '6+ years of experience as Data Scientist', 'evidence_count': 0}
  {'status': 'fail', 'rule': 'no fabricated skills (deterministic check)', 'claim': 'Removed skills not present in the source resume.', 'evidence_count': 1, 'removed_skills': ['Merchandising']}


**Interpretation:** the fix worked — the LLM tried to invent "Merchandising," and `_enforce_fidelity` caught and stripped it before delivery (confirmed above: it's absent from the final resume text and skills list). It only shows up in `validation_results.removed_skills`, System C's own transparency log of what it just caught. The grounding-gated judge is shown that entire JSON blob, including the audit trail, and misread the log entry as a live fabrication — capping C's score for a job where the delivered output was actually clean. This is a judge-methodology gap, not a regression in the system fix.

## Findings

**1. This is the hardest test case of the three, and it shows.**

Trixie's resume (real technical/business AI background) → C won 4.55 vs A/B's 2.85 post-fix. The fresh grad (some analytics coursework, pre-fix) → C won 4.2 vs 3.45/2.85. This resume (zero technical overlap anywhere) → C still wins post-fix (3.55 vs B's 3.35 vs A's 2.95), but by the narrowest margin of the three.

**2. The system-level fix did what it was supposed to do.**

The original DevOps Engineer fabrication ("linux," "python," "testing") is gone on retest — confirmed by inspecting the raw JSON, not just the score. `_enforce_fidelity` also caught the one thing the LLM tried to invent this round ("Merchandising," on a different job) before it ever reached the delivered resume text.

**3. But the retest surfaced a new, distinct problem: the judge scores the audit trail, not just the output.**

Because C's fix produces a visible record of what it caught (`validation_results.removed_skills`), and the judge is shown that entire record, a successfully-caught-and-removed fabrication can still drag down C's score — the opposite of what should happen. Fixing the system exposed a flaw in the evaluation harness, not a new flaw in the system.

**4. The judge's binary all-or-nothing rule (noted in the original run) is still a live issue.**

It still can't tell "invented and delivered" apart from "invented, caught, and removed." Both point the same direction: the next iteration of the judge should be shown only the final `rewritten_text`/`resume` fields, not the full internal JSON, and should weigh fabrication severity rather than applying one flat cap.

**5. As a Stage 7 failure case, now closed on the system side, open on the judge side:**

| Step | What happened |
|---|---|
| Input | Same A/B/C systems, tested on a candidate with zero technical/design overlap with any of the 5 jobs |
| Expected (original run) | C stays fabrication-free like it did on the fresh-grad resume |
| Actual (original run) | C stayed clean on 4/5 jobs, added 3 stray unverified words on DevOps Engineer |
| Fix (system) | Added `_enforce_fidelity`: a deterministic post-generation check stripping any skill not textually present in the source resume |
| Retest result | The original DevOps Engineer fabrication is gone — confirmed fixed. Overall score held flat (3.55 → 3.55) because the judge separately mis-flagged a successfully-removed item on a different job (Machine Learning), reading C's own removal audit trail as if it were live output |
| New finding (judge) | The grounding-gated judge should be shown only the delivered resume text, not System C's internal validation/audit JSON — otherwise a system getting *more* transparent about self-correction can score *worse* |